# Advanced OOP in Python - Solution

---

### About
This notebook applies object-oriented programming to a realistic data science workflow. We will build reusable components for loading delivery data, tracking model experiments, validating configuration values, composing workflows, and managing resources safely.

### Learning Objectives
- Implement inheritance, method overriding, and polymorphism.
- Distinguish instance methods, class methods, and static methods.
- Distinguish class attributes from instance attributes.
- Apply Python access conventions and properties.
- Use composition to connect workflow components.
- Use context managers for predictable setup and cleanup.

### Notebook Guide
- Scenario: FreshCart late delivery prediction
- Inheritance, overriding, and polymorphism
- Method types and class attributes
- Access conventions and properties
- Composition
- Context managers
- Want to know more: abstract classes, mixins, MRO, and destructors

# Scenario

You are a data scientist at **FreshCart**, an online grocery delivery company.

FreshCart wants to predict whether an order will arrive late so customer-support teams can intervene before a delivery issue occurs.

The workflow uses:
- CSV files containing historical orders;
- API responses containing live courier updates;
- model configuration values such as score thresholds;
- experiment tracking for model results;
- safe resource handling for files and connections.

Throughout this lesson, we will design Python classes that make this workflow easier to extend, test, and maintain.

In [ ]:
# Import the perf_counter function from Python’s built-in time module, which is useful for
# measuring how long code takes to run.

from time import perf_counter

# Inheritance, overriding, and polymorphism
---

A parent class can define a shared interface. Child classes inherit from the parent and provide their own specific behaviour.

In this workflow, every data source should know how to `.load()` records.

In [ ]:
class DataSource:
    def __init__(self, name):
        self.name = name

    def load(self):
        raise NotImplementedError("Subclasses must implement load().")

In [ ]:
try:
    DataSource(name="example").load()
except NotImplementedError as error:
    print(error)

In [ ]:
class CSVSource(DataSource):
    def __init__(self, name, rows):
        super().__init__(name)
        self.rows = rows

    def load(self):
        print(f"Loading historical orders from {self.name}")
        return self.rows

In [ ]:
historical_orders = [
    {
        "order_id": 101,
        "distance_km": 3.2,
        "prep_minutes": 18,
        "courier_delay_minutes": 4,
        "delivery_minutes": 34,
    },
    {
        "order_id": 102,
        "distance_km": 8.5,
        "prep_minutes": 27,
        "courier_delay_minutes": 16,
        "delivery_minutes": 58,
    },
]

CSVSource(
    name="orders.csv",
    rows=historical_orders,
).load()

In [ ]:
class APISource(DataSource):
    def __init__(self, name, response):
        super().__init__(name)
        self.response = response

    def load(self):
        print(f"Loading live courier updates from {self.name}")
        return self.response["results"]

In [ ]:
courier_response = {
    "status": "ok",
    "results": [
        {
            "order_id": 103,
            "distance_km": 5.1,
            "prep_minutes": 21,
            "courier_delay_minutes": 8,
            "delivery_minutes": 42,
        },
        {
            "order_id": 104,
            "distance_km": 10.4,
            "prep_minutes": 32,
            "courier_delay_minutes": 19,
            "delivery_minutes": 67,
        },
    ],
}

APISource(
    name="courier-api",
    response=courier_response,
).load()

## Polymorphism

Polymorphism means different objects can respond to the same method call. We can create a loop that does not need to know whether each object is a CSV source or an API source. It only needs each object to have a `.load()` method.

All data sources in this lesson return records with the same fields: `order_id`, `distance_km`, `prep_minutes`, `courier_delay_minutes`, and `delivery_minutes`. This shared record contract lets the workflow use different sources without source-specific logic.

In [ ]:
sources = [
    CSVSource("orders.csv", historical_orders),
    APISource("courier-api", courier_response),
]

for source in sources:
    records = source.load()
    print(records)

In [ ]:
# Check whether the first object is a DataSource instance
print(isinstance(sources[0], DataSource))

# Check whether CSVSource inherits from DataSource
print(issubclass(CSVSource, DataSource))

## Now You Try: Add another data source

Create `ManualEntrySource` so that it:
- Inherits from `DataSource`
- Stores a list of manually entered delivery records
- Implements `.load()`
- Works inside the same loop as the other sources
- Returns or prints a message showing what kind of data it loads

In [ ]:
class ManualEntrySource(DataSource):
    def __init__(self, name, records):
        # Initialise the parent class and store the delivery records
        super().__init__(name)
        self.records = records

    def load(self):
        # Display the source being loaded and return its records
        print(f"Loading manually entered delivery records from {self.name}")
        return self.records

In [ ]:
manual_source = ManualEntrySource(
    name="support-team-records",
    records=[
        {
            "order_id": 105,
            "distance_km": 2.7,
            "prep_minutes": 14,
            "courier_delay_minutes": 3,
            "delivery_minutes": 29,
        },
        {
            "order_id": 106,
            "distance_km": 7.4,
            "prep_minutes": 25,
            "courier_delay_minutes": 12,
            "delivery_minutes": 54,
        },
    ],
)

sources.append(manual_source)

for source in sources:
    print(source.load())

---

To the slides!

---

# Instance, class, and static methods
---

Classes can contain different kinds of methods.

- **Instance methods** receive `self` and work with one object.
- **Class methods** receive `cls` and work with the class itself.
- **Static methods** receive neither `self` nor `cls`; they are utility functions stored in the class namespace.

FreshCart wants to track model experiments. Each experiment has a model name and a validation score.

In [ ]:
class ExperimentRun:
    run_count = 0

    def __init__(self, model_name, score):
        self.model_name = model_name
        self.score = score
        ExperimentRun.run_count += 1

    def summary(self):
        return f"{self.model_name}: {self.score:.2f}"

    def __repr__(self):
        return f"ExperimentRun(model_name={self.model_name!r}, score={self.score!r})"

In [ ]:
# Instance method and class attribute
initial_run = ExperimentRun("logistic_regression", 0.81)
print(initial_run.summary())
print(ExperimentRun.run_count)

## Static methods

A static method receives neither `self` nor `cls`. It is useful for validation or other logic related to the class that does not need object or class state.

In [ ]:
class ExperimentRun:
    run_count = 0

    def __init__(self, model_name, score):
        self.model_name = model_name
        self.score = score
        ExperimentRun.run_count += 1

    def summary(self):
        return f"{self.model_name}: {self.score:.2f}"

    @staticmethod
    def is_valid_score(value):
        return 0 <= value <= 1

    def __repr__(self):
        return f"ExperimentRun(model_name={self.model_name!r}, score={self.score!r})"

In [ ]:
print(ExperimentRun.is_valid_score(0.86))
print(ExperimentRun.is_valid_score(1.2))

## Class methods

A class method receives `cls`. Here, `from_dict()` acts as an alternative constructor.

In [ ]:
class ExperimentRun:
    run_count = 0

    def __init__(self, model_name, score):
        self.model_name = model_name
        self.score = score
        ExperimentRun.run_count += 1

    def summary(self):
        return f"{self.model_name}: {self.score:.2f}"

    @staticmethod
    def is_valid_score(value):
        return 0 <= value <= 1

    @classmethod
    def from_dict(cls, data):
        return cls(data["model_name"], data["score"])

    def __repr__(self):
        return f"ExperimentRun(model_name={self.model_name!r}, score={self.score!r})"

In [ ]:
run_1 = ExperimentRun("logistic_regression", 0.81)
run_2 = ExperimentRun.from_dict({
    "model_name": "random_forest",
    "score": 0.86,
})

print(run_1.summary())
print(run_2.summary())
print(ExperimentRun.run_count)

## Class attributes vs instance attributes

`run_count` is a class attribute: it belongs to the class and is shared.

`model_name` and `score` are instance attributes: each experiment run has its own values.

In [ ]:
# Class attribute
print(ExperimentRun.run_count)

# Instance attributes
print(run_1.model_name)
print(run_2.model_name)

## Now You Try: Add an experiment label

Update or recreate `ExperimentRun` so that each run also stores a `dataset_name`.

Then:
- Update `.summary()` to include the data set
- Update `.from_dict()` to accept the data set
- Create two new experiment runs from dictionaries (one is given below - make up the other)

In [ ]:
class ExperimentRun:
    run_count = 0

    def __init__(self, model_name, score, dataset_name):
        self.model_name = model_name
        self.score = score
        self.dataset_name = dataset_name
        ExperimentRun.run_count += 1

    def summary(self):
        return (
            f"{self.model_name} on {self.dataset_name}: "
            f"{self.score:.2f}"
        )

    @staticmethod
    def is_valid_score(value):
        return 0 <= value <= 1

    @classmethod
    def from_dict(cls, data):
        return cls(
            data["model_name"],
            data["score"],
            data["dataset_name"]
        )

    def __repr__(self):
        return (
            f"ExperimentRun("
            f"model_name={self.model_name!r}, "
            f"score={self.score!r}, "
            f"dataset_name={self.dataset_name!r})"
        )

In [ ]:
run_1 = ExperimentRun.from_dict({
    "model_name": "gradient_boosting",
    "score": 0.88,
    "dataset_name": "freshcart_orders_v2"
})

run_2 = ExperimentRun.from_dict({
    "model_name": "random_forest",
    "score": 0.84,
    "dataset_name": "freshcart_orders_v3"
})

print(run_1.summary())
print(run_2.summary())
print(ExperimentRun.run_count)

---

To the slides!

---

# Access conventions and properties
---

Python does not enforce private and protected attributes in the same way as some other languages.

Common conventions:
- `name`: public;
- `_name`: internal use by convention;
- `__name`: name-mangled to reduce accidental clashes.

For validation, Python often uses **properties**.

FreshCart needs a model configuration object. The late-delivery threshold must stay between `0` and `1`.

In [ ]:
class ModelConfig:
    def __init__(self, model_name, late_threshold):
        self.model_name = model_name
        self.late_threshold = late_threshold

    @property
    def late_threshold(self):
        return self._late_threshold

    @late_threshold.setter
    def late_threshold(self, value):
        if not 0 <= value <= 1:
            raise ValueError("late_threshold must be between 0 and 1.")
        self._late_threshold = value

In [ ]:
config = ModelConfig("late_delivery_model", 0.65)
print(config.late_threshold)

config.late_threshold = 0.70
print(config.late_threshold)

In [ ]:
try:
    config.late_threshold = 1.4
except ValueError as error:
    print(error)

## Quick Check: Add another validated property

You have 5 minutes to add a `max_delivery_minutes` property to `ModelConfig`.

Requirements:
- It must be greater than `0`
- Invalid values should raise `ValueError`
- Valid values should be stored as `_max_delivery_minutes`

In [ ]:
# TODO: recreate ModelConfig with max_delivery_minutes.

# Example:
# config = ModelConfig("late_delivery_model", late_threshold=0.70, max_delivery_minutes=45)
# print(config.max_delivery_minutes)

In [ ]:
class ModelConfig:
    def __init__(self, model_name, late_threshold, max_delivery_minutes):
        self.model_name = model_name
        self.late_threshold = late_threshold
        self.max_delivery_minutes = max_delivery_minutes

    @property
    def late_threshold(self):
        return self._late_threshold

    @late_threshold.setter
    def late_threshold(self, value):
        if not 0 <= value <= 1:
            raise ValueError("late_threshold must be between 0 and 1")
        self._late_threshold = value

    @property
    def max_delivery_minutes(self):
        return self._max_delivery_minutes

    @max_delivery_minutes.setter
    def max_delivery_minutes(self, value):
        if value <= 0:
            raise ValueError("max_delivery_minutes must be greater than 0")
        self._max_delivery_minutes = value

In [ ]:
# Check that _max_delivery_minutes is stored correctly

config = ModelConfig(
    "late_delivery_model",
    late_threshold=0.70,
    max_delivery_minutes=45
)

print(config.max_delivery_minutes)

In [ ]:
try:
    config.max_delivery_minutes = 0
except ValueError as error:
    print(error)

# Composition
---

Composition means one object contains or uses another object.

A prediction workflow **has a** data source and **has a** model config.

This is different from inheritance. A workflow is not a data source; it uses one.

In [ ]:
class LateDeliveryWorkflow:
    def __init__(self, data_source, config):
        self.data_source = data_source
        self.config = config

    def run(self):
        records = self.data_source.load()
        predictions = []

        for record in records:
            risk_score = min(
                (record["distance_km"] / 10)
                + (record["prep_minutes"] / 60)
                + (record["courier_delay_minutes"] / 30),
                1,
            )

            predictions.append({
                "order_id": record["order_id"],
                "risk_score": round(risk_score, 2),
                "predict_late": risk_score >= self.config.late_threshold,
                "breaches_sla": (
                    record["delivery_minutes"]
                    > self.config.max_delivery_minutes
                ),
            })

        return predictions

In [ ]:
workflow = LateDeliveryWorkflow(
    data_source=CSVSource(
        name="orders.csv",
        rows=historical_orders,
    ),
    config=ModelConfig(
        model_name="late_delivery_model",
        late_threshold=0.65,
        max_delivery_minutes=45,
    ),
)

workflow.run()

## (If Time) Now You Try: Compose a different workflow

Create a workflow using a different data source and a different threshold.

Then answer:
- Did you need to change the `LateDeliveryWorkflow` class?
- Why is that useful?

In [ ]:
# TODO: create a new workflow with another DataSource and ModelConfig.

In [ ]:
manual_source = ManualEntrySource(
    name="support-team-records",
    records=[
        {
            "order_id": 201,
            "distance_km": 4.4,
            "prep_minutes": 20,
            "courier_delay_minutes": 7,
            "delivery_minutes": 36,
        },
        {
            "order_id": 202,
            "distance_km": 9.3,
            "prep_minutes": 30,
            "courier_delay_minutes": 18,
            "delivery_minutes": 61,
        },
    ],
)

new_config = ModelConfig(
    model_name="late_delivery_model",
    late_threshold=0.60,
    max_delivery_minutes=40,
)

new_workflow = LateDeliveryWorkflow(
    data_source=manual_source,
    config=new_config,
)

new_workflow.run()

**Sample answers:**

- No, the LateDeliveryWorkflow class did not need to change.
- This is useful because composition lets us swap compatible data sources and configurations without rewriting the workflow itself.

---

To the slides!

---

# Context Managers
---

Some resources need setup and cleanup: files, database connections, API sessions, timers, and cloud resources.

A context manager lets us use `with` so cleanup happens predictably.

In [ ]:
class WorkflowTimer:
    def __init__(self, label):
        self.label = label

    def __enter__(self):
        self.start = perf_counter()
        print(f"Starting {self.label}")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        elapsed = perf_counter() - self.start
        print(f"Finished {self.label} in {elapsed:.4f} seconds")

In [ ]:
with WorkflowTimer("late delivery workflow"):
    predictions = workflow.run()

predictions

## Now You Try: Build a Managed Resource

Create `DataConnection` as a context manager.

It should:
- Print `Opening connection` in `__enter__`
- Print `Closing connection` in `__exit__`
- Return itself from `__enter__`
- Have a `.query()` method that returns sample delivery records

In [ ]:
class DataConnection:
    def __enter__(self):
        # TODO: open the connection
        pass

    def __exit__(self, exc_type, exc_value, traceback):
        # TODO: close the connection
        pass

    def query(self):
        # TODO: return sample records
        pass

In [ ]:
# Uncomment after completing DataConnection.
# with DataConnection() as connection:
#     print(connection.query())

In [ ]:
class DataConnection:
    def __enter__(self):
        print("Opening connection")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print("Closing connection")

    def query(self):
        return [
            {
                "order_id": 301,
                "distance_km": 6.2,
                "prep_minutes": 23,
                "courier_delay_minutes": 9,
                "delivery_minutes": 46,
            },
            {
                "order_id": 302,
                "distance_km": 11.1,
                "prep_minutes": 35,
                "courier_delay_minutes": 21,
                "delivery_minutes": 71,
            },
        ]

In [ ]:
with DataConnection() as connection:
    print(connection.query())

# Conclusions and Takeaways
---

- Inheritance is useful when one class is a specialised version of another.
- Polymorphism lets different objects share the same interface.
- Instance, class, and static methods solve different problems.
- Class attributes are shared; instance attributes belong to one object.
- Python access control is mostly convention-based.
- Properties are useful for validation.
- Composition often keeps workflows flexible.
- Context managers are preferred for predictable resource cleanup.

# Want to Know More? The next things to think about...
---

The sections below are useful next steps if you want to dig deeper into OOP in Python.

## Abstract base classes

An abstract base class can enforce that subclasses implement required methods.

In [ ]:
from abc import ABC, abstractmethod

class StrictDataSource(ABC):
    @abstractmethod
    def load(self):
        pass

In [ ]:
class StrictCSVSource(StrictDataSource):
    def __init__(self, rows):
        self.rows = rows

    def load(self):
        return self.rows

StrictCSVSource([{"order_id": 201}]).load()

## Mixins and MRO

A mixin adds a focused capability. MRO, or Method Resolution Order, is the order Python follows when looking for methods across multiple parent classes.

In [ ]:
class LoggingMixin:
    def load(self):
        print("Logging before load")
        return super().load()

class LoggedCSVSource(LoggingMixin, CSVSource):
    pass

logged_source = LoggedCSVSource("logged-orders.csv", historical_orders)
logged_source.load()

In [ ]:
[cls.__name__ for cls in LoggedCSVSource.mro()]

## Destructors

Python has a `__del__()` method, but it is rarely the best tool for important cleanup. The timing can be hard to reason about. Prefer context managers when cleanup matters.

In [ ]:
class DemoDestructor:
    def __del__(self):
        print("Object is being cleaned up")

obj = DemoDestructor()
del obj